# Imports 

In [0]:
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score


# Load Data

In [0]:

# Define your catalog, schema, and table names
CATALOG_NAME = "ml_catalog"
SCHEMA_NAME = "titanic_schema"
TRAIN_TABLE_NAME = "train"
TEST_TABLE_NAME = "test"
TEST_LABEL_TABLE_NAME = "gender_submission"

# Load the data from Unity Catalog 
# For this lab, we will work with pandas so it is easier
train = spark.table(f"{CATALOG_NAME}.{SCHEMA_NAME}.{TRAIN_TABLE_NAME}").toPandas()
test = spark.table(f"{CATALOG_NAME}.{SCHEMA_NAME}.{TEST_TABLE_NAME}").toPandas()
test_label = spark.table(f"{CATALOG_NAME}.{SCHEMA_NAME}.{TEST_LABEL_TABLE_NAME}").toPandas()

Let's view the train, test, and label data that we are working with

In [0]:
print(f"train shape: {train.shape}")
train.head()

In [0]:
print(f"test_feats shape: {test.shape}")
test.head()

In [0]:
print(f"test_y_true shape: {test_label.shape}")
test_label.head()

# Feature Engineering

In [0]:
# For this lab, these are the columns we are going to work with
features = ["Fare", "Age", "Pclass", "SibSp", "Parch", "Sex"]

# Check For NaN Value
train[features].isna().sum()

There is a lot of NULL features in the Age columns. So let's fix it

In [0]:
# Building the Feature Engineering Pipeline
continuous_features = ["Fare", "Age", "Pclass", "SibSp", "Parch"]
categorical_features = ["Sex"]

continuous_transformer = SimpleImputer(strategy="mean")
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # This is the safer option to use if you are working with OneHotEncoder. This will introduce Bias, but it is the best option we have. 
    ("encoder", OneHotEncoder(drop="first")) # drop='first' so that OneHotEncoder will only create n-1 column for n unique values in the column. Avoid collinearity
])

preprocessor = ColumnTransformer(
    transformers=[
        ('continuous', continuous_transformer, continuous_features),
        ('categorical', categorical_transformer, categorical_features)
    ]
)

Even though other features does not contain NULL values, it is best practice to have an impute strategy for every column you are working with. We cannot 100% guarantee that test dataset will also be devoid of any null values on these columns.

Now, for the time being, let's make sure that the NULL values in Age is being handled properly and validate that other imputation method are working is intended

In [0]:
# First: Fit the column transformer
train_feats = train[[col for col in features if col in train.columns]]
preprocessor.fit(train_feats)

In [0]:
# Second: Let's transform the data using the fitted preprocessor and check if the values of the NULL "Age" column is imputed
processed_train_feats = preprocessor.transform(train_feats)

processed_train_feats = pd.DataFrame(processed_train_feats, columns=preprocessor.get_feature_names_out())
processed_train_feats.isna().sum()

All Age NULL values have been succesfully imputed. Next, let's further validate, if the imputation score is correct

In [0]:
train_feats[train_feats.isna().any(axis=1)]

In [0]:
processed_train_feats.iloc[5]

In [0]:
processed_train_feats.iloc[19]

Good, Age has been succesfully imputed and Sex becomes a binary column 1 if male and 0 if female.

# Model Training

In [0]:
# Let's start back with our original_variables

# First redefine the features (optional, so it is easier to follow in this lab). 
features = ["Fare", "Age", "Pclass", "SibSp", "Parch", "Sex"] 

# Split train data into X and y
X = train[features]
y = train["Survived"]

# Split train data into train and eval data 
X_train, X_eval, y_train, y_eval = train_test_split(X, y, test_size=0.2, random_state=42)


**We already have a test dataset, why do we still split train into train and eval? Why not evaluate on the test dataset?**

A: It is industry practice that test dataset is use once, discard forever. If you evaluate all of your trained model on your test dataset, your model generalizability power is biased to the test dataset. You want to train a best model that performs well on your evaluation dataset, and then estimate it's performance when deployed for real-world usecase by using the test dataset. It is to this point that test dataset cannot be used more than once.

In [0]:
# Build our model
rf_classifier = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

# Define our end-to-end scikit-learn pipeline
ml_pipeline = Pipeline(steps=[
    ("feature_engineering", preprocessor),
    ("classifier", rf_classifier)
])

#Track Model Training in MLFlow

In [0]:
ml_pipeline.fit(X_train, y_train)

In [0]:
# These are packages we are using, it is already imported at the top, but we import it again for clarity
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

username = "millikanevan@gmail.com" # Change to your username
mlflow.set_experiment(f"/Users/{username}/lab1_experiment")

mlflow.sklearn.autolog(
    log_input_examples = True,
    silent = True
)

with mlflow.start_run(run_name="v0_1_0") as run:
    print("Training model...")
    ml_pipeline.fit(X_train, y_train)

    y_pred_proba = ml_pipeline.predict_proba(X_eval)[:, 1]
    

    # By turning proba into binary prediction using this method, we have better fine grained control on the threshold
    threshold = 0.5
    y_pred = (y_pred_proba > threshold).astype(int)

    auc_metric = roc_auc_score(y_eval, y_pred_proba)
    f1_score = f1_score(y_eval, y_pred)
    accuracy = accuracy_score(y_eval, y_pred)

    mlflow.log_metric("eval_auc", auc_metric)
    mlflow.log_metric("eval_f1_score", f1_score)
    mlflow.log_metric("eval_accuracy", accuracy)
    
    mlflow.sklearn.log_model(
        ml_pipeline,
        artifact_path = "model-artifacts",
        input_example = X_train[:5],
        signature = infer_signature(X_train, y_train)
    )

    model_uri = f"runs:/{run.info.run_id}/model-artifacts"


    

# Hyperparameter Tuning and Comparing Result in MLFlow

In [0]:
# These are packages we are using, it is already imported at the top, but we import it again for clarity
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from sklearn.model_selection import GridSearchCV

username = "millikanevan@gmail.com"
mlflow.set_experiment(f"/Users/{username}/lab1_experiment")

# Move autolog out from start_run()
# This will automatically capture the GridSearchCV child runs
mlflow.sklearn.autolog(
    log_input_examples = True,
    silent = True
)

# 2. Define your hyperparameter grid
# NOTE: Use 'classifier__' prefix because the RF model is named 'classifier' in your ml_pipeline
param_grid = {
    "classifier__n_estimators": [50, 100, 150],
    "classifier__max_depth": [3, 5, 7]
}

# 3. Wrap your pipeline in GridSearchCV
grid_search = GridSearchCV(
    estimator=ml_pipeline,
    param_grid=param_grid,
    cv=3,                 # 3-fold cross validation
    scoring="roc_auc",    # Optimize for AUC
    n_jobs=-1             # Use all available processors
)

with mlflow.start_run(run_name="v0_2_0") as run:
    print("Training grid search models... (this will create nested child runs)")
    
    # This single fit command triggers MLflow to log all parameter combinations!
    grid_search.fit(X_train, y_train)
    
    print(f"Best parameters found: {grid_search.best_params_}")
    
    # GridSearchCV automatically refits the best model on the entire X_train set.
    # We can extract it to evaluate it on our custom X_eval set.
    best_model = grid_search.best_estimator_

    y_pred_proba = best_model.predict_proba(X_eval)[:, 1]


    # By turning proba into binary prediction using this method, we have better fine grained control on the threshold
    threshold = 0.5
    y_pred = (y_pred_proba > threshold).astype(int)

    eval_auc = roc_auc_score(y_eval, y_pred_proba)
    eval_f1_score = f1_score(y_eval, y_pred)
    eval_accuracy = accuracy_score(y_eval, y_pred)

    mlflow.log_metric("eval_auc", eval_auc)
    mlflow.log_metric("eval_f1_score", eval_f1_score)
    mlflow.log_metric("eval_accuracy", eval_accuracy)
    
    mlflow.sklearn.log_model(
        ml_pipeline,
        artifact_path = "model-artifacts",
        input_example = X_train[:5],
        signature = infer_signature(X_train, y_train)
    )

    model_uri = f"runs:/{run.info.run_id}/best_estimator"
    print(f"Best model logged to URI: {model_uri}")


    


Now, if you go the experiments tab, you'll find v0.2.0 with many nested run, and the v0.2.0 itself contains the artifact of the best model

In [0]:
# "databricks-uc" is the registry URI for Unity Catalog
mlflow.set_registry_uri("databricks-uc")

CATALOG_NAME = "ml_catalog"
SCHEMA_NAME = "titanic_schema"
model_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.hyperparamgrid_model"

registered_model = mlflow.register_model(model_uri=model_uri, name=model_name)

In [0]:
# Additional - You can manually created nested run like this...
# Useful is you want to manually try out different hyperparameter configuration 

# import mlflow

# # 1. Start the Parent Run
# with mlflow.start_run(run_name="Manual_Hyperparameter_Search") as parent_run:
#     print(f"Parent Run ID: {parent_run.info.run_id}")
    
#     # You can log metrics/params to the parent run here if needed
    
#     # 2. Start a Child Run (Notice nested=True)
#     with mlflow.start_run(run_name="Config_1_Depth_3", nested=True):
#         mlflow.log_param("max_depth", 3)
#         mlflow.log_metric("auc", 0.75)
#         print("Finished Config 1")

#     # 3. Start another Child Run
#     with mlflow.start_run(run_name="Config_2_Depth_5", nested=True):
#         mlflow.log_param("max_depth", 5)
#         mlflow.log_metric("auc", 0.82)
#         print("Finished Config 2")


# Evaluating on Test Dataset

In [0]:
# Create the test features and test label
test_joined = pd.merge(test, test_label, on=['PassengerId'])
X_test = test_joined[features]
y_test = test_joined['Survived']

In [0]:
# Assume that you have trained a few model and have decided on an earlier version of the model that yields the best performance
# Let's reload that run and evaluate it for the final time on the test dataset

# Copy this from the experiments tab
target_run_id = "e57018a8f0ca4a44af72a8e067f2eba8"

model_uri = f"runs:/{target_run_id}/model-artifacts"

print(f"Loading model from run {target_run_id}")
loaded_model_pipeline = mlflow.sklearn.load_model(model_uri)

print("Performing predictions on test dataset...")
y_test_pred_proba = loaded_model_pipeline.predict_proba(X_test)[:, 1]

# Apply your custom threshold
threshold = 0.5
y_test_pred = (y_test_pred_proba > threshold).astype(int)

# 4. Calculate the final test metrics
test_auc = roc_auc_score(y_test, y_test_pred_proba)
test_f1 = f1_score(y_test, y_test_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Test AUC: {test_auc:.4f}")

# Resume the original run and log the new metrics
# By passing the run_id parameter, MLflow knows NOT to create a new run, 
# but to append these metrics to the existing one.
with mlflow.start_run(run_id=target_run_id):
    
    # Prefixing with 'test_' clearly separates these from your 'eval_' metrics in the UI
    mlflow.log_metric("test_auc", test_auc)
    mlflow.log_metric("test_f1_score", test_f1)
    mlflow.log_metric("test_accuracy", test_accuracy)
    
    print("Successfully logged final test metrics to the original run!")

# Deploying Model to Unity Catalog

 Next, after we created a model and want to deploy it. We just need to register the model into our unity catalog

In [0]:
# "databricks-uc" is the registry URI for Unity Catalog
mlflow.set_registry_uri("databricks-uc")

CATALOG_NAME = "ml_catalog"
SCHEMA_NAME = "titanic_schema"
model_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.example_titanic_model"

# Select your run ID from the experiments tab
run_id = "e57018a8f0ca4a44af72a8e067f2eba8"
model_uri = f"runs:/{run_id}/model-artifacts"

registered_model = mlflow.register_model(model_uri=model_uri, name=model_name)


You can go to "Catalog" tab and see that under "titanic_schema" there is a new directory called titanic_model

In [0]:
from mlflow.tracking.client import MlflowClient

client = MlflowClient()

client.set_registered_model_alias(
    name = registered_model.name,
    alias = "dev",
    version = registered_model.version
)

# Example to alias your model version manually
# client.set_registered_model_alias(
#     name = "ml_catalog.titanic_schema.titanic_model",
#     alias = "dev",
#     version = "1"
# )

Navigate to the "Catalog" tab and you'll see that the model of your choosing is now tagged under @`dev`

Alias are unique per registered model_name. If you train another model and then tag the version 2 of the model to dev, the dev tag will be deleted from version 1

In [0]:
# Try running this again and go to Catalog tab and see that the alias for dev has moved from version 1 to version 2
model_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.example_titanic_model"

registered_model = mlflow.register_model(model_uri=model_uri, name=model_name)


client.set_registered_model_alias(
    name = model_name,
    alias = "dev",
    version = "2"
)

Aliases are helpful to assign a single named reference to your model. In case you upgrade your model, the alias name doesn't change. External users will still refer to the same model alias even though the model version has been upgraded


Next, we'll cover how to compare experiments in the experiments tab


# Side Example: Using Databricks Document Processing to Turn Unstructured Data into Useful Features

In our train, we have a column called "name" that contain some useful information. However, by traditional method, turning this column into usable feature is hard. 

In this side example, we will demonstrate how to use ai_classify to classify name to "parent", "child", "adult traveling alone", "adult traveling with family" or "unkown" by comparing each row's name to the list of names on the passengers list

In [0]:

# Define your catalog, schema, and table names
CATALOG_NAME = "ml_catalog"
SCHEMA_NAME = "titanic_schema"
TRAIN_TABLE_NAME = "train"
TEST_TABLE_NAME = "test"
TEST_LABEL_TABLE_NAME = "gender_submission"

query = f"""
WITH passenger_names AS (
    SELECT collect_list(name) AS all_names
    FROM {CATALOG_NAME}.{SCHEMA_NAME}.{TRAIN_TABLE_NAME}
)
SELECT
    t.*,
    p.all_names,
    ai_classify(
        t.name,
        ARRAY('parent', 'child', 'adult traveling alone', 'adult traveling with family', 'unknown')
    ) AS classification
FROM {CATALOG_NAME}.{SCHEMA_NAME}.{TRAIN_TABLE_NAME} t
CROSS JOIN passenger_names p
LIMIT 10
"""

train_with_ai_classify = spark.sql(query).toPandas()

In [0]:
train_with_ai_classify

You can view more intelligent document processing functions that Databricks provided here: https://docs.databricks.com/aws/en/agents/agent-bricks/intelligent-document-processing
